In [ ]:
"""
PFD-Net V1 PINN — 二维 Poisson 方程求解（无监督，三阶段 + 自适应域分解）
=========================================================================
PDE:   -Δu(x,y) = f(x,y),   (x,y) ∈ Ω = [-1,1]²
边界:   u = g_b(x,y),        (x,y) ∈ ∂Ω   (Dirichlet)

精确解: F(x,y) = g(x) + g(y),  g(x) = exp(-x²) sin(μx²)
源项:   f(x,y) = -(g''(x) + g''(y))
边界值: g_b = F(x,y)|∂Ω

网络结构:
  低频 APINN: 固定傅里叶嵌入（低频频率）+ tanh MLP，全域
  高频 APINN: 固定傅里叶嵌入（高频频率）+ tanh MLP，凸包子域加权叠加
  PFDNet = 低频 APINN + Σ φ_i(x) · 高频 APINN_i

三阶段训练（全程无监督 PINN）:
  Phase 1 — 低频 APINN 预训练:
      仅训练 apinn_low，PDE残差 + BC 损失
  Phase 2 — 高频 APINN 预热（冻结低频）:
      冻结 apinn_low，仅训练各子域高频网络
      域分解指标改为无监督：PDE残差 + 梯度模 + |Δu|
  Phase 3 — 全参数联合训练:
      解冻所有参数，PDE残差 + BC 损失

损失函数（各阶段统一）:
    L = λ_r · MSE[ -(u_xx + u_yy) - f ]  +  λ_b · MSE[ u|∂Ω - g_b ]

域分解（无监督）:
    综合指标 = w0·||PDE残差|| + w1·||∇u|| + w2·||Δu||
    → 分位数阈值 → DBSCAN → 凸包 → ConvexHullSubdomain
"""

import os
import time
import warnings
import numpy as np
import torch
import torch.nn as nn
from matplotlib.path import Path as MplPath
from scipy.spatial import ConvexHull
from sklearn.cluster import DBSCAN

warnings.filterwarnings("ignore")

# ── 设备 ──────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")
torch.manual_seed(42)
np.random.seed(42)

CHECKPOINT_DIR = "checkpoints_pfdv1_pinn_poisson"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

MU       = 30.0
LAMBDA_R = 1.0
LAMBDA_B = 10.0
pi       = torch.tensor(np.pi, dtype=torch.float64, device=device)


# ==============================================================================
# 精确解 & 源项
# ==============================================================================

def g_func(x: torch.Tensor) -> torch.Tensor:
    return torch.exp(-x ** 2) * torch.sin(MU * x ** 2)


def g_second(x: torch.Tensor) -> torch.Tensor:
    ex      = torch.exp(-x ** 2)
    s       = torch.sin(MU * x ** 2)
    c       = torch.cos(MU * x ** 2)
    coeff_s = 4 * x**2 - 4 * MU**2 * x**2 - 2
    coeff_c = 2 * MU - 8 * MU * x**2
    return ex * (coeff_s * s + coeff_c * c)


def u_exact(xy: torch.Tensor) -> torch.Tensor:
    return g_func(xy[:, 0:1]) + g_func(xy[:, 1:2])


def f_source(xy: torch.Tensor) -> torch.Tensor:
    """源项 f，满足 -Δu = f"""
    return -(g_second(xy[:, 0:1]) + g_second(xy[:, 1:2]))


def g_boundary(xy: torch.Tensor) -> torch.Tensor:
    return u_exact(xy)


# ==============================================================================
# 网络结构
# ==============================================================================

class FixedFourierEmbed2D(nn.Module):
    """固定傅里叶嵌入（频率不参与训练），输出维度 = 4 * len(scales)"""
    def __init__(self, scales):
        super().__init__()
        self.register_buffer(
            "scales",
            torch.tensor(scales, dtype=torch.float64)
        )

    def get_scales(self):
        return self.scales.detach().cpu().numpy()

    def forward(self, xy: torch.Tensor) -> torch.Tensor:
        feats = []
        for s in self.scales:
            feats += [
                torch.sin(2 * pi * s * xy[:, 0:1]),
                torch.cos(2 * pi * s * xy[:, 0:1]),
                torch.sin(2 * pi * s * xy[:, 1:2]),
                torch.cos(2 * pi * s * xy[:, 1:2]),
            ]
        return torch.cat(feats, dim=-1)


class APINN2D(nn.Module):
    """统一 APINN 网络（低频/高频通用，固定傅里叶频率 + tanh MLP）"""
    def __init__(self, hidden_dim: int = 128, num_layers: int = 4,
                 scales=(1, 2, 4)):
        super().__init__()
        self.embed  = FixedFourierEmbed2D(scales)
        in_dim      = 2 + 4 * len(scales)
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(in_dim, hidden_dim))
        for _ in range(num_layers - 2):
            self.layers.append(nn.Linear(hidden_dim, hidden_dim))
        self.layers.append(nn.Linear(hidden_dim, 1))
        for m in self.layers:
            nn.init.xavier_normal_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(self, xy: torch.Tensor) -> torch.Tensor:
        h = torch.cat([xy, self.embed(xy)], dim=-1)
        for layer in self.layers[:-1]:
            h = torch.tanh(layer(h))
        return self.layers[-1](h)


# ==============================================================================
# 凸包子域（支撑函数）
# ==============================================================================

class ConvexHullSubdomain:
    """
    封装单个凸包子域：计算到凸包边界的有符号距离，并构建光滑支撑函数。
    支撑函数：
      内部 → sigmoid(decay · d) ≈ 1
      外部 → exp(decay · d)    → 0
    """
    def __init__(self, hull_vertices: np.ndarray, decay: float = 15.0):
        self.vertices_np = hull_vertices
        self.decay       = decay

        verts = np.vstack([hull_vertices, hull_vertices[0]])
        edges = verts[1:] - verts[:-1]
        norms = np.linalg.norm(edges, axis=1, keepdims=True)
        self.edge_dirs  = edges / (norms + 1e-12)
        self.edge_start = verts[:-1]
        self.edge_len   = norms.flatten()
        self.mpl_path   = MplPath(hull_vertices)
        self.center     = hull_vertices.mean(axis=0)
        self.radius     = np.max(np.linalg.norm(hull_vertices - self.center, axis=1))

    def dist_to_hull_boundary(self, xy_np: np.ndarray) -> np.ndarray:
        N = len(xy_np)
        min_dist = np.full(N, np.inf)
        for k in range(len(self.edge_dirs)):
            p0   = self.edge_start[k]
            d    = self.edge_dirs[k]
            L    = self.edge_len[k]
            vec  = xy_np - p0
            t    = np.clip(vec @ d, 0, L)
            proj = p0 + t[:, None] * d
            dist = np.linalg.norm(xy_np - proj, axis=1)
            min_dist = np.minimum(min_dist, dist)
        inside = self.mpl_path.contains_points(xy_np).astype(float)
        return np.where(inside > 0.5, min_dist, -min_dist)

    def support_function(self, xy: torch.Tensor) -> torch.Tensor:
        xy_np = xy.detach().cpu().numpy()
        d_np  = self.dist_to_hull_boundary(xy_np).astype(np.float64)
        d_t   = torch.tensor(d_np, dtype=torch.float64,
                              device=xy.device).unsqueeze(-1)
        inside = (d_t > 0).float()
        phi = (inside * torch.sigmoid(self.decay * d_t)
               + (1 - inside) * torch.exp(self.decay * d_t))
        return phi.clamp(0.0, 1.0)


class PFDNet2D(nn.Module):
    """低频 APINN + 多凸包子域高频 APINN（支撑函数加权叠加）"""
    def __init__(self, apinn_low: APINN2D, subdomains: list,
                 high_scales, hidden_dim: int = 128, num_layers: int = 4):
        super().__init__()
        self.apinn_low  = apinn_low
        self.subdomains = subdomains
        self.high_nets  = nn.ModuleList([
            APINN2D(hidden_dim, num_layers, high_scales)
            for _ in subdomains
        ])

    def forward(self, xy: torch.Tensor) -> torch.Tensor:
        u = self.apinn_low(xy)
        for sd, net in zip(self.subdomains, self.high_nets):
            phi = sd.support_function(xy)   # (N,1)，不可微，无需反传
            u   = u + phi * net(xy)
        return u

    def print_scales(self, prefix: str = ""):
        print(f"{prefix}低频APINN 固定频率: "
              f"{np.round(self.apinn_low.embed.get_scales(), 2)}")
        for i, net in enumerate(self.high_nets):
            print(f"{prefix}高频APINN[{i}] 固定频率: "
                  f"{np.round(net.embed.get_scales(), 2)}")


# ==============================================================================
# 自动微分：Δu = u_xx + u_yy
# ==============================================================================

def laplacian(model: nn.Module, xy: torch.Tensor) -> torch.Tensor:
    """
    xy: (N,2)，调用前已设 requires_grad=True
    返回: Δu = u_xx + u_yy，shape (N,1)
    """
    u = model(xy)

    grad_u = torch.autograd.grad(
        u, xy,
        grad_outputs=torch.ones_like(u),
        create_graph=True,
        retain_graph=True,
    )[0]                                        # (N,2)

    u_xx = torch.autograd.grad(
        grad_u[:, 0:1], xy,
        grad_outputs=torch.ones_like(grad_u[:, 0:1]),
        create_graph=True,
        retain_graph=True,
    )[0][:, 0:1]                                # (N,1)

    u_yy = torch.autograd.grad(
        grad_u[:, 1:2], xy,
        grad_outputs=torch.ones_like(grad_u[:, 1:2]),
        create_graph=True,
        retain_graph=True,
    )[0][:, 1:2]                                # (N,1)

    return u_xx + u_yy


# ==============================================================================
# PINN 损失计算（各阶段复用）
# ==============================================================================

def pinn_loss(model: nn.Module,
              xy_int: torch.Tensor,
              f_int:  torch.Tensor,
              xy_bc:  torch.Tensor,
              u_bc:   torch.Tensor,
              lambda_r: float,
              lambda_b: float):
    """
    xy_int: 普通张量（函数内部 clone + requires_grad_(True)）
    f_int:  源项，已 detach
    xy_bc:  边界点，普通张量
    u_bc:   边界真值，已 detach
    返回: (total, loss_r, loss_b)
    """
    # PDE 残差
    xy_r  = xy_int.clone().requires_grad_(True)
    lap_u = laplacian(model, xy_r)
    loss_r = torch.mean((-lap_u - f_int) ** 2)

    # 边界条件
    loss_b = nn.functional.mse_loss(model(xy_bc), u_bc)

    total = lambda_r * loss_r + lambda_b * loss_b
    return total, loss_r, loss_b


# ==============================================================================
# 采样
# ==============================================================================

def sample_interior(n: int, seed: int = 1) -> torch.Tensor:
    """[-1,1]² 内部混合采样（均匀网格 + 随机），返回普通张量"""
    rng  = np.random.RandomState(seed)
    side = int(np.sqrt(n // 2))
    gx   = np.linspace(-1 + 1e-4, 1 - 1e-4, side)
    gy   = np.linspace(-1 + 1e-4, 1 - 1e-4, side)
    GX, GY  = np.meshgrid(gx, gy)
    xy_grid = np.stack([GX.ravel(), GY.ravel()], axis=1)
    n_rand  = n - xy_grid.shape[0]
    xy_rand = rng.uniform(-1.0, 1.0, size=(n_rand, 2))
    pts     = np.concatenate([xy_grid, xy_rand], axis=0)
    return torch.tensor(pts, dtype=torch.float64, device=device)


def sample_boundary(n_per_edge: int, seed: int = 2) -> torch.Tensor:
    """四条边各 n_per_edge 个点，返回 (4*n_per_edge, 2)"""
    rng    = np.random.RandomState(seed)
    t      = rng.uniform(-1.0, 1.0, n_per_edge)
    bottom = np.stack([t,  -np.ones(n_per_edge)], axis=1)
    top    = np.stack([t,   np.ones(n_per_edge)], axis=1)
    left   = np.stack([-np.ones(n_per_edge), t],  axis=1)
    right  = np.stack([ np.ones(n_per_edge), t],  axis=1)
    pts    = np.concatenate([bottom, top, left, right], axis=0)
    return torch.tensor(pts, dtype=torch.float64, device=device)


# ==============================================================================
# 无监督域分解：综合指标 = PDE残差 + 梯度模 + |Δu|
# ==============================================================================

def _norm01(arr: np.ndarray) -> np.ndarray:
    return (arr - arr.min()) / (arr.max() - arr.min() + 1e-10)


def compute_pinn_indicators(model: nn.Module,
                             xy_int: torch.Tensor,
                             f_int:  torch.Tensor):
    """
    无监督域分解指标（不依赖精确解）：
      - PDE 残差：|-Δu - f|
      - 梯度模：  ||∇u||
      - Laplacian：|Δu|
    返回三个 numpy (N,) 数组
    """
    xy = xy_int.detach().clone().requires_grad_(True)
    u  = model(xy)

    grad_u = torch.autograd.grad(
        u, xy,
        grad_outputs=torch.ones_like(u),
        create_graph=True,
        retain_graph=True,
    )[0]                                        # (N,2)
    grad_norm = torch.norm(grad_u, dim=-1)      # (N,)

    ones = torch.ones(len(xy), dtype=torch.float64, device=device)

    u_xx = torch.autograd.grad(
        grad_u[:, 0], xy,
        grad_outputs=ones,
        retain_graph=True,
        create_graph=False,
    )[0][:, 0]                                  # (N,)

    u_yy = torch.autograd.grad(
        grad_u[:, 1], xy,
        grad_outputs=ones,
        retain_graph=False,
        create_graph=False,
    )[0][:, 1]                                  # (N,)

    lap   = u_xx + u_yy                         # (N,)
    pde_r = torch.abs(-lap - f_int.squeeze(-1)) # |-Δu - f|
    lap_a = torch.abs(lap)                      # |Δu|

    return (pde_r.detach().cpu().numpy(),
            grad_norm.detach().cpu().numpy(),
            lap_a.detach().cpu().numpy())


def identify_subdomains(
    model:              nn.Module,
    xy_int:             torch.Tensor,
    f_int:              torch.Tensor,
    composite_weights:  tuple = (1.0, 0.8, 0.5),
    percentile:         int   = 70,
    eps:                float = 0.15,
    min_samples:        int   = 20,
    max_subdomains:     int   = 6,
    noise_ratio:        float = 0.08,
    decay:              float = 15.0,
) -> list:
    """
    无监督域分解流程：
      1. 计算 PDE残差 / 梯度模 / |Δu| 三类指标
      2. 加权合成 → 分位数阈值筛选高频候选点
      3. DBSCAN 聚类（自动扩大 eps 重试）
      4. 每个聚类做 ConvexHull → ConvexHullSubdomain
    """
    print(f"\n{'='*60}")
    print(f"  域分解（无监督：PDE残差 + 梯度 + Laplacian → DBSCAN → 凸包）")
    print(f"{'='*60}")

    pde_r_np, grad_np, lap_np = compute_pinn_indicators(model, xy_int, f_int)

    comp = (composite_weights[0] * _norm01(pde_r_np)
            + composite_weights[1] * _norm01(grad_np)
            + composite_weights[2] * _norm01(lap_np))

    threshold = np.percentile(comp, percentile)
    mask      = comp > threshold
    xy_np     = xy_int.detach().cpu().numpy()
    xy_high   = xy_np[mask]

    print(f"  高频候选点: {mask.sum()} / {len(comp)}"
          f"（第{percentile}百分位，阈值={threshold:.4f}）")

    if len(xy_high) < min_samples:
        print("  候选点不足，退回全域单子域")
        verts = np.array([[-1, -1], [1, -1], [1, 1], [-1, 1]], dtype=float)
        return [ConvexHullSubdomain(verts, decay)]

    # 自动扩大 eps 重试（最多 4 次）
    labels, cur_eps = None, eps
    unique = []
    for _ in range(4):
        labels = DBSCAN(eps=cur_eps, min_samples=min_samples).fit(xy_high).labels_
        unique = sorted(set(labels) - {-1})
        n_noise = int((labels == -1).sum())
        print(f"  DBSCAN eps={cur_eps:.3f} → {len(unique)} 个聚类，噪声 {n_noise}")
        if unique:
            break
        cur_eps *= 2.0

    if not unique:
        print("  聚类失败，退回全域单子域")
        verts = np.array([[-1, -1], [1, -1], [1, 1], [-1, 1]], dtype=float)
        return [ConvexHullSubdomain(verts, decay)]

    # 按大小排序，过滤噪声簇，取前 max_subdomains
    sizes = {lb: (labels == lb).sum() for lb in unique}
    max_s = max(sizes.values())
    valid = [lb for lb in unique if sizes[lb] >= max_s * noise_ratio]
    valid.sort(key=lambda lb: -sizes[lb])
    valid = valid[:max_subdomains]
    for lb in valid:
        print(f"  ✓ 聚类 {lb}: n={sizes[lb]}")

    subdomains = []
    for lb in valid:
        pts = xy_high[labels == lb]
        if len(pts) < 3:
            continue
        try:
            hull  = ConvexHull(pts)
            verts = pts[hull.vertices]
            sd    = ConvexHullSubdomain(verts, decay)
            subdomains.append(sd)
            cx, cy = verts.mean(axis=0)
            print(f"  凸包子域: {len(pts)} 点，顶点 {len(verts)}，"
                  f"中心=({cx:.2f},{cy:.2f})")
        except Exception as e:
            print(f"  凸包构建失败 ({e})，跳过")

    if not subdomains:
        print("  所有凸包失败，退回全域单子域")
        verts = np.array([[-1, -1], [1, -1], [1, 1], [-1, 1]], dtype=float)
        subdomains = [ConvexHullSubdomain(verts, decay)]

    print(f"  共生成 {len(subdomains)} 个凸包子域")
    return subdomains


# ==============================================================================
# 三阶段训练（全程无监督 PINN）
# ==============================================================================

def train_pfdv1_pinn(
    # 网络超参
    hidden_dim      : int   = 128,
    num_layers      : int   = 4,
    low_scales              = (1, 2, 4),
    high_scales             = (8, 16, 32),
    # 配点
    n_interior      : int   = 10000,
    n_per_edge      : int   = 250,
    # 训练轮次（总 15000 epoch，分三阶段）
    pretrain_epochs : int   = 8000,
    warmup_epochs   : int   = 2000,
    joint_epochs    : int   = 5000,
    # 学习率
    lr_pretrain     : float = 5e-3,
    lr_warmup       : float = 1e-3,
    lr_joint        : float = 5e-4,
    # 损失权重
    lambda_r        : float = LAMBDA_R,
    lambda_b        : float = LAMBDA_B,
    # 域分解超参
    composite_weights       = (1.0, 0.8, 0.5),
    percentile      : int   = 70,
    eps_dbscan      : float = 0.15,
    min_samples     : int   = 20,
    max_subdomains  : int   = 6,
    noise_ratio     : float = 0.08,
    decay           : float = 15.0,
    # 日志
    log_every       : int   = 1,
):
    total_epochs = pretrain_epochs + warmup_epochs + joint_epochs
    # assert total_epochs == 15000, \
    #     f"三阶段之和应为 15000，当前为 {total_epochs}"

    # ── 固定配点 ──────────────────────────────────────────────────────────────
    xy_int = sample_interior(n_interior, seed=1)    # 普通张量
    f_int  = f_source(xy_int).detach()              # 源项，固定

    xy_bc  = sample_boundary(n_per_edge, seed=2)
    u_bc   = g_boundary(xy_bc).detach()

    print(f"内部配点: {xy_int.shape[0]},  边界配点: {xy_bc.shape[0]}")

    # ── 测试集（100×100 均匀网格，与精确解对比）────────────────────────────
    nx      = 100
    xv, yv  = np.meshgrid(np.linspace(-1, 1, nx), np.linspace(-1, 1, nx))
    xy_test = torch.tensor(
        np.stack([xv.ravel(), yv.ravel()], axis=1),
        dtype=torch.float64, device=device
    )
    u_test         = u_exact(xy_test).detach()
    u_test_sq_mean = torch.mean(u_test ** 2).item()

    def eval_l2(model: nn.Module) -> float:
        with torch.no_grad():
            pred = model(xy_test)
            return torch.sqrt(
                torch.mean((pred - u_test) ** 2) / u_test_sq_mean
            ).item()

    # ── 全局记录 ──────────────────────────────────────────────────────────────
    epochs_record = []
    res_losses    = []
    bc_losses     = []
    total_losses  = []
    l2_errors     = []
    epoch_offset  = 0

    def record_and_print(ep_local, total_l, loss_r, loss_b, model, tag):
        ep_global = ep_local + epoch_offset
        l2        = eval_l2(model)
        epochs_record.append(ep_global)
        res_losses.append(loss_r.item())
        bc_losses.append(loss_b.item())
        total_losses.append(total_l.item())
        l2_errors.append(l2)
        if ep_local % max(1, log_every * 5) == 0 or ep_local == 1:
            print(
                f"  {tag} {ep_local:6d} | "
                f"res: {loss_r.item():.3e} | "
                f"bc: {loss_b.item():.3e} | "
                f"total: {total_l.item():.3e} | "
                f"L2: {l2:.6f} | {time.time()-t0:.1f}s"
            )

    t0 = time.time()

    # ==========================================================================
    # Phase 1：低频 APINN 预训练
    # ==========================================================================
    print(f"\n{'='*60}")
    print(f"  Phase 1: 低频APINN 预训练（PINN）| {pretrain_epochs} epochs")
    print(f"  低频频率: {list(low_scales)}  lr={lr_pretrain:.2e}")
    print(f"  λ_r={lambda_r}, λ_b={lambda_b}")
    print(f"{'='*60}")

    apinn_low = APINN2D(hidden_dim, num_layers, low_scales).double().to(device)
    total_p1  = sum(p.numel() for p in apinn_low.parameters())
    print(f"低频APINN 参数量: {total_p1:,}")

    opt1 = torch.optim.Adam(apinn_low.parameters(), lr=lr_pretrain)
    # sch1 = torch.optim.lr_scheduler.CosineAnnealingLR(
    #     opt1, pretrain_epochs, eta_min=1e-5)

    for ep in range(1, pretrain_epochs + 1):
        apinn_low.train()
        opt1.zero_grad()
        loss, loss_r, loss_b = pinn_loss(
            apinn_low, xy_int, f_int, xy_bc, u_bc, lambda_r, lambda_b)
        loss.backward()
        nn.utils.clip_grad_norm_(apinn_low.parameters(), 1.0)
        opt1.step()
        # sch1.step()

        if ep % log_every == 0 or ep == 1:
            record_and_print(ep, loss, loss_r, loss_b, apinn_low, "pretrain")

    torch.save(apinn_low.state_dict(),
               os.path.join(CHECKPOINT_DIR, "phase1_apinn_low.pt"))
    print(f"\n  Phase 1 结束 | L2={eval_l2(apinn_low):.6f}")
    epoch_offset += pretrain_epochs

    # ==========================================================================
    # 无监督域分解（利用 Phase 1 低频 APINN 的 PDE 残差等指标）
    # ==========================================================================
    subdomains = identify_subdomains(
        apinn_low, xy_int, f_int,
        composite_weights=composite_weights,
        percentile=percentile,
        eps=eps_dbscan,
        min_samples=min_samples,
        max_subdomains=max_subdomains,
        noise_ratio=noise_ratio,
        decay=decay,
    )

    # ==========================================================================
    # Phase 2：高频 APINN 预热（冻结低频）
    # ==========================================================================
    print(f"\n{'='*60}")
    print(f"  Phase 2: 高频APINN 预热（冻结低频）| {warmup_epochs} epochs")
    print(f"  子域数: {len(subdomains)}  高频频率: {list(high_scales)}  lr={lr_warmup:.2e}")
    print(f"{'='*60}")

    pfdnet   = PFDNet2D(apinn_low, subdomains, high_scales,
                        hidden_dim, num_layers).double().to(device)
    total_p2 = sum(p.numel() for p in pfdnet.parameters())
    print(f"PFDNet 总参数量: {total_p2:,}")

    # 冻结低频参数
    for p in pfdnet.apinn_low.parameters():
        p.requires_grad = False

    high_params = list(pfdnet.high_nets.parameters())

    opt2 = torch.optim.Adam(high_params, lr=lr_warmup)
    # sch2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    #     opt2, warmup_epochs, eta_min=1e-5)

    for ep in range(1, warmup_epochs + 1):
        pfdnet.train()
        opt2.zero_grad()
        loss, loss_r, loss_b = pinn_loss(
            pfdnet, xy_int, f_int, xy_bc, u_bc, lambda_r, lambda_b)
        loss.backward()
        nn.utils.clip_grad_norm_(high_params, 1.0)
        opt2.step()
        # sch2.step()

        if ep % log_every == 0 or ep == 1:
            record_and_print(ep, loss, loss_r, loss_b, pfdnet, "warmup ")

    torch.save(pfdnet.state_dict(),
               os.path.join(CHECKPOINT_DIR, "phase2_pfdnet.pt"))
    print(f"\n  Phase 2 结束 | L2={eval_l2(pfdnet):.6f}")
    epoch_offset += warmup_epochs

    # ==========================================================================
    # Phase 3：全参数联合训练
    # ==========================================================================
    print(f"\n{'='*60}")
    print(f"  Phase 3: 全参数联合训练（PINN）| {joint_epochs} epochs")
    print(f"  lr={lr_joint:.2e}")
    print(f"{'='*60}")

    # 解冻低频参数
    for p in pfdnet.apinn_low.parameters():
        p.requires_grad = True

    opt3 = torch.optim.Adam([
        {'params': pfdnet.apinn_low.parameters(), 'lr': lr_joint},
        {'params': pfdnet.high_nets.parameters(), 'lr': lr_joint},
    ])
    # sch3 = torch.optim.lr_scheduler.CosineAnnealingLR(
    #     opt3, joint_epochs, eta_min=1e-6)

    for ep in range(1, joint_epochs + 1):
        pfdnet.train()
        opt3.zero_grad()
        loss, loss_r, loss_b = pinn_loss(
            pfdnet, xy_int, f_int, xy_bc, u_bc, lambda_r, lambda_b)
        loss.backward()
        nn.utils.clip_grad_norm_(list(pfdnet.parameters()), 1.0)
        opt3.step()
        # sch3.step()

        if ep % log_every == 0 or ep == 1:
            record_and_print(ep, loss, loss_r, loss_b, pfdnet, "joint  ")

    # ── 最终指标 ──────────────────────────────────────────────────────────────
    with torch.no_grad():
        fp   = pfdnet(xy_test)
        fl2  = torch.sqrt(
            torch.mean((fp - u_test) ** 2) / u_test_sq_mean).item()
        fmae = torch.mean(torch.abs(fp - u_test)).item()
    total_time = time.time() - t0

    print(f"\n{'='*60}")
    print(f"  训练完成 | L2: {fl2:.6f} | MAE: {fmae:.6f} | 时间: {total_time:.1f}s")
    pfdnet.print_scales("  ")
    print(f"{'='*60}")

    # ── 保存完整模型（包括子域坐标）──────────────────────────────────────────────────
    # 1. 保存模型权重
    torch.save(pfdnet.state_dict(),
               os.path.join(CHECKPOINT_DIR, "pfdv1_pinn_weights.pt"))
    
    # 2. 保存子域凸包顶点坐标（用循环逐个保存，不要用 np.savez 直接存列表）
    for i, sd in enumerate(subdomains):
        np.save(os.path.join(CHECKPOINT_DIR, f"subdomain_{i}_vertices.npy"), 
                sd.vertices_np)
    np.save(os.path.join(CHECKPOINT_DIR, "n_subdomains.npy"), 
            len(subdomains))
    
    # 3. 保存训练曲线
    np.savez(
        os.path.join(CHECKPOINT_DIR, "pfdv1_pinn_curves.npz"),
        epochs     = np.array(epochs_record),
        res_loss   = np.array(res_losses),
        bc_loss    = np.array(bc_losses),
        total_loss = np.array(total_losses),
        l2_error   = np.array(l2_errors),
    )
    
    # 4. 保存完整检查点
    full_checkpoint = {
        'model_state': pfdnet.state_dict(),
        'subdomain_vertices': [sd.vertices_np for sd in subdomains],
        'hidden_dim': hidden_dim,
        'num_layers': num_layers,
        'low_scales': low_scales,
        'high_scales': high_scales,
        'final_l2': fl2,
        'final_mae': fmae,
        'total_time': total_time,
    }
    torch.save(full_checkpoint, os.path.join(CHECKPOINT_DIR, "pfdv1_full.pt"))
    
    print(f"完整模型已保存至 {CHECKPOINT_DIR}/")
    print(f"  - 模型权重: pfdv1_pinn_weights.pt")
    print(f"  - 子域坐标: subdomain_*_vertices.npy ({len(subdomains)} 个子域)")
    print(f"  - 完整检查点: pfdv1_full.pt")

# ==============================================================================
if __name__ == "__main__":
    train_pfdv1_pinn(
        hidden_dim      = 128,
        num_layers      = 4,
        low_scales      = (1, 2, 4),
        high_scales     = (8, 16, 32),
        n_interior      = 10000,
        n_per_edge      = 250,
        pretrain_epochs = 8000,
        warmup_epochs   = 2000,
        joint_epochs    = 5000,
        lr_pretrain     = 5e-3,
        lr_warmup       = 1e-3,
        lr_joint        = 5e-4,
        lambda_r        = 1.0,
        lambda_b        = 10.0,
        composite_weights = (0.1, 1.0, 1.0),
        percentile      = 70,
        eps_dbscan      = 0.15,
        min_samples     = 20,
        max_subdomains  = 6,
        noise_ratio     = 0.08,
        decay           = 15.0,
        log_every       = 1,
    )